# US-009：Source Localization 入门

**目标：** 了解 EEG 源定位的基本概念，用 fsaverage 模板跑通源重建流程。

**核心流程：** 前向模型 → 逆问题求解 → 源空间可视化

## 1. 基本概念

### Sensor Space vs Source Space
```
Sensor Space (头皮)          Source Space (大脑皮层)
┌─────────────────┐          ┌─────────────────┐
│  EEG 通道 (60)  │    →     │  几千个皮层顶点  │
│  电位分布       │  逆问题  │  电流偶极子分布  │
└─────────────────┘          └─────────────────┘
```

### 前向模型 (Forward Model)
已知源求头皮电位：$X = G \cdot S$
- **G**：正向矩阵 (gain matrix)，描述每个源如何传播到每个通道
- 需要：头模型 (BEM) + 源空间 (皮层网格) + 通道位置

### 逆问题 (Inverse Problem)
已知头皮电位求源：$S = G^{-1} \cdot X$ → 但 G 不是方阵，是病态的！
需要用正则化方法求解：MNE / dSPM / sLORETA / eLORETA

## 2. 准备数据

用 MNE sample 数据演示（它自带 Evoked 对象）。

In [ ]:
import mne
import numpy as np
import os.path as op

# 加载 sample 数据
sample_dir = mne.datasets.sample.data_path()
subjects_dir = op.join(sample_dir, 'subjects')  # 头部模型目录

raw_fname = op.join(sample_dir, 'MEG', 'sample', 'sample_audvis_raw.fif')
raw = mne.io.read_raw_fif(raw_fname, preload=True)
raw.pick_types(eeg=True, stim=True)
raw.filter(l_freq=1.0, h_freq=40.0)

# 事件和 Epochs
events = mne.find_events(raw, stim_channel='STI 014')
event_id = {'auditory/left': 1, 'visual/left': 3}
epochs = mne.Epochs(
    raw, events, event_id, tmin=-0.2, tmax=0.5,
    baseline=(None, 0), preload=True, reject=dict(eeg=150e-6)
)
evoked_aud = epochs['auditory/left'].average()
evoked_vis = epochs['visual/left'].average()

print(f"Evoked 就绪: {evoked_aud}")

## 3. 计算前向解 (Forward Solution)

### 3.1 使用 fsaverage 模板（无需个人 MRI）

MNE 内置 fsaverage 模板大脑，无需 MRI 即可做源重建。

In [ ]:
# 下载 fsaverage 模板（首次运行需下载，约 400MB）
fs_dir = mne.datasets.fetch_fsaverage(verbose=True)
subjects_dir = op.dirname(fs_dir)
subject = 'fsaverage'

print(f"fsaverage 路径: {fs_dir}")

### 3.2 源空间 (Source Space)

源空间是大脑皮层上的顶点网格，每个顶点是一个可能的源。

In [ ]:
# 创建源空间：'ico-4' 表示 ~2562 个顶点/半球
src = mne.setup_source_space(
    subject, spacing='ico4',
    subjects_dir=subjects_dir,
    add_dist=False,
)
print(f"源空间: {src}")
print(f"左半球顶点: {len(src[0]['vertno'])}, 右半球: {len(src[1]['vertno'])}")

### 3.3 前向解

计算每个源到每个通道的传播矩阵 G。

In [ ]:
# 使用三层壳 BEM 模型（fsaverage 自带）
# 注意：纯 EEG 情况下可以用更简单的球模型
conductivity = (0.3,)  # 单层（大脑电导率）
model = mne.make_bem_model(
    subject, ico=4,
    conductivity=conductivity,
    subjects_dir=subjects_dir,
)
bem = mne.make_bem_solution(model)

# 计算前向解
fwd = mne.make_forward_solution(
    evoked_aud.info,           # 通道信息
    trans='fsaverage',         # 使用 fsaverage 默认 trans（通道→头部对齐）
    src=src,
    bem=bem,
    eeg=True,
    mindist=5.0,               # 源到内壳的最小距离 (mm)
    n_jobs=1,
)
print(f"前向解: {fwd}")
print(f"Gain 矩阵形状: {fwd['sol']['data'].shape}")  # (n_channels, n_sources)

## 4. 逆问题求解

三种常用算法：

In [ ]:
# 估计噪声协方差（从 baseline 时段）
noise_cov = mne.compute_covariance(
    epochs, tmin=-0.2, tmax=0.0,
    method='empirical',
)

# ---- 方法 1: MNE (最小范数估计) ----
inv_mne = mne.minimum_norm.make_inverse_operator(
    evoked_aud.info, fwd, noise_cov,
    loose=0.2,           # 源朝向约束（0=严格径向，1=自由朝向）
    depth=0.8,           # 深度加权
)

# 应用逆算子
stc_mne = mne.minimum_norm.apply_inverse(
    evoked_aud, inv_mne,
    lambda2=1.0 / 9.0,   # 正则化参数（SNR 估计 = 3 → λ²=1/SNR²）
    method='MNE',
)
print(f"MNE STC: {stc_mne}")

In [ ]:
# ---- 方法 2: dSPM (动态统计参数映射) ----
stc_dspm = mne.minimum_norm.apply_inverse(
    evoked_aud, inv_mne,
    lambda2=1.0 / 9.0,
    method='dSPM',
)
print(f"dSPM STC: {stc_dspm}")

In [ ]:
# ---- 方法 3: sLORETA (标准化低分辨率电磁断层成像) ----
stc_sloreta = mne.minimum_norm.apply_inverse(
    evoked_aud, inv_mne,
    lambda2=1.0 / 9.0,
    method='sLORETA',
)
print(f"sLORETA STC: {stc_sloreta}")

### 三种方法对比

| 方法 | 输出单位 | 特点 |
|------|---------|------|
| MNE | 电流密度 (A·m) | 最平滑，定位偏浅 |
| dSPM | Z-score (无量纲) | 深度归一化，适合统计 |
| sLORETA | Z-score | 零定位误差（理想情况），最常用 |

## 5. 可视化源活动

### 5.1 在标准脑上绘制

In [ ]:
# 使用 fsaverage 的大脑
brain = stc_sloreta.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    hemi='both',             # 左右半球
    views='lateral',         # 侧视图
    time_viewer=True,        # 交互式时间浏览
    initial_time=0.1,        # 初始显示的时间点
    clim='auto',
    background='white',
)

### 5.2 峰值时刻的源分布

In [ ]:
# 找到 N100 (~100ms) 时的源活动
peak_time = 0.1  # 100 ms

# 提取峰值时刻的数据
stc_peak = stc_sloreta.copy().crop(tmin=peak_time, tmax=peak_time)

# 绘制
brain = stc_peak.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    hemi='both',
    views=['lateral', 'medial', 'dorsal', 'ventral'],
    background='white',
)

### 5.3 条件间源空间对比

听觉 vs 视觉：听觉应有颞叶激活，视觉应有枕叶激活。

In [ ]:
# 计算视觉的源
stc_vis = mne.minimum_norm.apply_inverse(
    evoked_vis, inv_mne,
    lambda2=1.0 / 9.0,
    method='sLORETA',
)

# 差异：听觉 - 视觉
stc_diff = stc_sloreta - stc_vis

brain = stc_diff.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    hemi='both',
    initial_time=0.1,
    time_viewer=True,
    background='white',
)

## 6. 简化流程：只用球模型

不需要 BEM/freesurfer，用简单的多层球模型也能做近似源定位：

In [ ]:
# ===== 简化的球模型流程 =====
# 1. 用球模型替代 BEM
# sphere = mne.make_sphere_model(r0='auto', head_radius='auto')

# 2. 源空间
# src = mne.setup_volume_source_space(
#     subject=None, pos=15.0,  # 体积源空间
#     sphere=sphere,
# )

# 3. 前向解
# fwd = mne.make_forward_solution(
#     evoked.info, trans=None, src=src, bem=sphere,
#     eeg=True, mindist=5.0,
# )

print("球模型简化流程需求较少，适合快速原型。")
print("但定位精度不如 BEM。")
print("详情参见: https://mne.tools/stable/auto_tutorials/forward/30_forward.html")

## 7. 源定位 vs 传感器分析

| 维度 | Sensor Space | Source Space |
|------|-------------|-------------|
| 分辨率 | 厘米级（容积传导模糊） | 毫米级（理论） |
| 数据量 | 64-256 通道 | 5000-20000 个源 |
| 计算量 | 低 | 高 |
| 对解剖模型依赖 | 无（只需蒙太奇） | 高（需要 MRI/BEM） |
| 适用 | ERP/时频/连通性 | 定位功能区/手术规划 |

## 8. 展望

- **个人 MRI + FreeSurfer**：把 fsaverage 换成你自己的大脑
- **eLORETA**：比 sLORETA 更准确（MNE 1.0+ 支持）
- **Beamformers (LCMV/DICS)**：空间滤波方法，适合频域分析
- **混合 MEG+EEG**：利用两种模态的互补信息

## 9. 练习

1. 对比 MNE / dSPM / sLORETA 在同一时间点的源分布
2. 调整 `loose` 参数（0 到 1），观察源朝向对结果的影响
3. 用 `stc.extract_label_time_course()` 提取特定脑区的时间序列